# Supplementary Figure S3 - Null Validation

Two independent null-model controls for the yeast GI-PCC reference clustering (`linkage_method="ward"`, `linkage_metric="euclidean"`, `linkage_threshold=16`, `min_cluster_size=30`, `min_overlap=2`, global Benjamini-Hochberg FDR, `qval <= 0.05`; 7 clusters, 709 enriched rows, 331 significant cluster-term pairs): an annotation-label permutation null and a same-size cluster-randomization null. Both address whether the observed enrichment could arise from arbitrary labels or arbitrary cluster partitions rather than genuine structure.

In [ ]:
# Enable inline plotting for notebooks
%matplotlib inline

In [ ]:
# Figure export utilities (PNG, 300 DPI).
from pathlib import Path
import warnings

import matplotlib.pyplot as plt

warnings.filterwarnings(
    "ignore",
    message=r"Dropped .* annotations after matrix filtering.*",
    category=RuntimeWarning,
)

PNG_DIR = Path("png") / "supp_fig_3"
PNG_DIR.mkdir(parents=True, exist_ok=True)


def save_figure_png(name, *, fig=None, dpi=300, pad_inches=0.02):
    """Save a Matplotlib figure as a high-quality PNG in the notebook export folder."""
    if fig is None:
        fig = plt.gcf()
    output_path = PNG_DIR / f"{name}.png"
    fig.savefig(
        output_path,
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=pad_inches,
        facecolor=fig.get_facecolor(),
        edgecolor="none",
    )
    print(f"Saved {output_path}")
    return output_path

## Run Reference Analysis

Rerun the fig_1 yeast GI-PCC reference clustering and GO Biological Process enrichment directly
from raw repo inputs (`data/yeast/gi_pcc_sampled.tsv`, `data/yeast/go_bp_name_to_orfs.json`),
using the same reference configuration as `fig_1.ipynb` and `supp_fig_1.ipynb`. Both null
controls below reuse this single clustering result and rerun enrichment per replicate.

In [ ]:
import json

import pandas as pd

import himalayas
from himalayas import Analysis, Annotations, Matrix

DATA_DIR = Path("data/yeast")
GO_BP_PATH = DATA_DIR / "go_bp_name_to_orfs.json"
MATRIX_PATH = DATA_DIR / "gi_pcc_sampled.tsv"

with GO_BP_PATH.open("r", encoding="utf-8") as fh:
    go_bp = json.load(fh)

DF_GI_PCC = pd.read_csv(MATRIX_PATH, sep="\t", index_col=0)

print(f"HiMaLAYAS version: {himalayas.__version__}")

REFERENCE_CONFIG = dict(
    linkage_method="ward",
    linkage_metric="euclidean",
    linkage_threshold=16,
    optimal_ordering=True,
    min_cluster_size=30,
)
MIN_OVERLAP = 2
QVAL_CUTOFF = 0.05
FDR_SCOPE = "global"

matrix = Matrix(DF_GI_PCC)
annotations = Annotations(go_bp, matrix)

analysis = (
    Analysis(matrix, annotations)
    .cluster(**REFERENCE_CONFIG)
    .enrich(min_overlap=MIN_OVERLAP)
    .finalize(col_cluster=True, fdr_scope=FDR_SCOPE)
)
results = analysis.results
results_sig = results.filter(f"qval <= {QVAL_CUTOFF}")

assert (len(results.df), len(results_sig.df), len(results.clusters.cluster_sizes)) == (
    709,
    331,
    7,
), "GI-PCC reference clustering result changed from expected publication values"

n_observed_sig = len(results_sig.df)

print(f"All enriched rows: {len(results.df):,}")
print(f"Significant rows (q<={QVAL_CUTOFF}): {n_observed_sig:,}")
print(f"Clusters: {len(results.clusters.cluster_sizes)}")

## Annotation-Label Permutation Null

Keep the observed reference clustering fixed. For each of `N_REPLICATES` replicates, apply a
random bijection over the matrix gene universe to relabel which genes carry which GO
Biological Process term membership, preserving every term's size exactly. Rebuild `Annotations`
from the permuted mapping and rerun enrichment (`run_cluster_hypergeom`) against the fixed
reference clusters, followed by global Benjamini-Hochberg FDR (`with_qvalues(method=FDR_SCOPE)`).
Clustering is not rerun; only enrichment reruns per replicate.

In [ ]:
import numpy as np
from himalayas.core.enrichment import run_cluster_hypergeom

N_REPLICATES = 1000
PERMUTATION_SEED = 20260809

# Prefilter GO-BP annotations to the matrix gene universe before permutation.
matrix_gene_set = set(matrix.labels)
go_bp_matrix = {term: [g for g in genes if g in matrix_gene_set] for term, genes in go_bp.items()}

perm_rng = np.random.default_rng(PERMUTATION_SEED)
gene_universe = matrix.labels

perm_counts = []
for _ in range(N_REPLICATES):
    shuffled = perm_rng.permutation(gene_universe)
    relabel_map = dict(zip(gene_universe, shuffled))
    permuted_go_bp = {term: [relabel_map[g] for g in genes] for term, genes in go_bp_matrix.items()}
    perm_annotations = Annotations(permuted_go_bp, matrix)
    perm_null_results = run_cluster_hypergeom(
        matrix, results.clusters, perm_annotations, min_overlap=MIN_OVERLAP
    ).with_qvalues(method=FDR_SCOPE)
    n_sig = (
        0
        if perm_null_results.df.empty
        else int((perm_null_results.df["qval"] <= QVAL_CUTOFF).sum())
    )
    perm_counts.append(n_sig)

perm_null_dist = pd.DataFrame({"n_sig": perm_counts})
perm_null_max = int(perm_null_dist["n_sig"].max())
perm_empirical_p = (int((perm_null_dist["n_sig"].to_numpy() >= n_observed_sig).sum()) + 1) / (
    N_REPLICATES + 1
)
perm_row = {
    "observed_value": n_observed_sig,
    "null_max": perm_null_max,
    "empirical_pvalue": perm_empirical_p,
}

print(f"Annotation-label permutation null ({N_REPLICATES} replicates, seed={PERMUTATION_SEED}):")
print(f"  observed significant cluster-term pairs : {perm_row['observed_value']}")
print(f"  null max                                 : {perm_row['null_max']}")
print(f"  empirical p-value                        : {perm_row['empirical_pvalue']:.6f}")

## Same-Size Random-Cluster Null

Keep the GO Biological Process annotations fixed. For each of `N_REPLICATES` replicates,
randomly partition the matrix gene universe into clusters that preserve the observed cluster
count and sizes (`results.clusters.cluster_sizes`), without rerunning hierarchical clustering.
Rerun enrichment against the fixed annotations, followed by global Benjamini-Hochberg FDR, exactly
as in the permutation null above.

In [ ]:
from types import SimpleNamespace

RANDOM_CLUSTER_SEED = 20260810

rand_rng = np.random.default_rng(RANDOM_CLUSTER_SEED)
cluster_sizes = results.clusters.cluster_sizes
cluster_ids_ordered = list(cluster_sizes.keys())

rand_counts = []
for _ in range(N_REPLICATES):
    shuffled = rand_rng.permutation(gene_universe)
    cluster_to_labels = {}
    offset = 0
    for cid in cluster_ids_ordered:
        size = cluster_sizes[cid]
        cluster_to_labels[cid] = set(shuffled[offset : offset + size])
        offset += size

    rand_clusters = SimpleNamespace(
        unique_clusters=np.array(cluster_ids_ordered),
        cluster_to_labels=cluster_to_labels,
        cluster_sizes=dict(cluster_sizes),
        merge_small_clusters=results.clusters.merge_small_clusters,
        min_cluster_size=results.clusters.min_cluster_size,
    )

    rand_null_results = run_cluster_hypergeom(
        matrix, rand_clusters, annotations, min_overlap=MIN_OVERLAP
    ).with_qvalues(method=FDR_SCOPE)
    n_sig = (
        0
        if rand_null_results.df.empty
        else int((rand_null_results.df["qval"] <= QVAL_CUTOFF).sum())
    )
    rand_counts.append(n_sig)

rand_null_dist = pd.DataFrame({"n_sig": rand_counts})
rand_null_max = int(rand_null_dist["n_sig"].max())
rand_empirical_p = (int((rand_null_dist["n_sig"].to_numpy() >= n_observed_sig).sum()) + 1) / (
    N_REPLICATES + 1
)
rand_row = {
    "observed_value": n_observed_sig,
    "null_max": rand_null_max,
    "empirical_pvalue": rand_empirical_p,
}

print(f"Same-size random-cluster null ({N_REPLICATES} replicates, seed={RANDOM_CLUSTER_SEED}):")
print(f"  observed significant cluster-term pairs : {rand_row['observed_value']}")
print(f"  null max                                 : {rand_row['null_max']}")
print(f"  empirical p-value                        : {rand_row['empirical_pvalue']:.6f}")

## Export Individual Panels

Export Panels A and B as standalone PNGs for manual figure assembly in PowerPoint. Each panel is
a compact null histogram with the observed value shown on the same transformed x-axis. The x
positions use `log10(significant_pairs + 1)`, which preserves the zero-valued null replicates
while making the distance from the null range (0-2) to the observed value (331) visible without
an axis break.

In [ ]:
LABEL_COLOR = "black"
FONT = "Helvetica"
plt.rcParams["font.family"] = FONT
OBSERVED_COLOR = "#d73027"
NULL_COLOR = "#4c78a8"
BAR_EDGE = "#5b5b5b"
GRID_COLOR = "#cecece"
SPINE_COLOR = "#333333"
PANEL_FIGSIZE = (5.5, 4.8)

TITLE_FONTSIZE = 16
TITLE_PAD = 14
AXIS_LABEL_FONTSIZE = 12
TICK_LABEL_FONTSIZE = 11
TICK_PARAMS_LABELSIZE = 11
BAR_LABEL_FONTSIZE = 10
OBSERVED_LABEL_FONTSIZE = 12
STATS_LABEL_FONTSIZE = 11

BAR_WIDTH = 0.1
BAR_LINEWIDTH = 1
BAR_LABEL_Y_MULT = 1.15
OBSERVED_LINEWIDTH = 2
OBSERVED_LABEL_X_OFFSET = 0.1
ANNOTATION_ANCHOR_Y = 300
ANNOTATION_LINE_GAP_PT = 8
GRID_LINEWIDTH = 1
GRID_ALPHA = 0.8
SPINE_LINEWIDTH = 0.5

XLIM_LEFT = -0.15
XLIM_RIGHT_PAD = 0.15
YLIM_BOTTOM = 0.8
YLIM_TOP = 1500

null_panels = [
    {
        "panel": "A",
        "name": "panel_a_permutation_null",
        "title": "Annotation-label permutation",
        "dist": perm_null_dist,
        "observed": int(perm_row["observed_value"]),
        "null_max": int(perm_row["null_max"]),
        "empirical_p": float(perm_row["empirical_pvalue"]),
        "x_values": sorted(perm_null_dist["n_sig"].unique().tolist()),
    },
    {
        "panel": "B",
        "name": "panel_b_random_cluster_null",
        "title": "Same-size random clusters",
        "dist": rand_null_dist,
        "observed": int(rand_row["observed_value"]),
        "null_max": int(rand_row["null_max"]),
        "empirical_p": float(rand_row["empirical_pvalue"]),
        "x_values": sorted(rand_null_dist["n_sig"].unique().tolist()),
    },
]

assert all(panel["observed"] == n_observed_sig for panel in null_panels)

x_tick_values = sorted(set([0, 1, 2, 10, 100, n_observed_sig]))


def x_transform(values):
    values = np.asarray(values, dtype=float)
    return np.log10(values + 1.0)


def plot_null_panel(ax, panel):
    """Render one null-histogram panel (bars + observed marker) onto ax."""
    counts = (
        panel["dist"]["n_sig"].value_counts().reindex(panel["x_values"], fill_value=0).sort_index()
    )
    x_raw = counts.index.to_numpy()
    x_pos = x_transform(x_raw)
    y = counts.to_numpy()

    ax.bar(
        x_pos,
        y,
        width=BAR_WIDTH,
        color=NULL_COLOR,
        edgecolor=BAR_EDGE,
        linewidth=BAR_LINEWIDTH,
        align="center",
        zorder=3,
    )
    for xi, yi in zip(x_pos, y):
        ax.text(
            xi,
            yi * BAR_LABEL_Y_MULT,
            f"{yi}",
            ha="center",
            va="bottom",
            fontsize=BAR_LABEL_FONTSIZE,
            color=LABEL_COLOR,
            fontname=FONT,
        )

    obs_x = float(x_transform([panel["observed"]])[0])
    ax.axvline(obs_x, color=OBSERVED_COLOR, linewidth=OBSERVED_LINEWIDTH, zorder=4)

    # One right-aligned annotation block anchored near the observed line: the
    # "Observed" line and the "largest null / p" line are stacked by a fixed
    # point-space offset (not data-space) so they read as one statistical
    # summary regardless of where they land on the log y-axis.
    anchor_xy = (obs_x - OBSERVED_LABEL_X_OFFSET, ANNOTATION_ANCHOR_Y)
    ax.annotate(
        f"Observed = {panel['observed']}",
        xy=anchor_xy,
        xycoords="data",
        ha="right",
        va="bottom",
        fontsize=OBSERVED_LABEL_FONTSIZE,
        color=OBSERVED_COLOR,
        fontname=FONT,
    )
    ax.annotate(
        f"largest null = {panel['null_max']}, empirical p = {panel['empirical_p']:.3f}",
        xy=anchor_xy,
        xytext=(0, -ANNOTATION_LINE_GAP_PT),
        textcoords="offset points",
        xycoords="data",
        ha="right",
        va="top",
        fontsize=STATS_LABEL_FONTSIZE,
        color=LABEL_COLOR,
        fontname=FONT,
    )

    ax.set_title(
        panel["title"],
        loc="left",
        fontsize=TITLE_FONTSIZE,
        fontname=FONT,
        color=LABEL_COLOR,
        pad=TITLE_PAD,
    )
    ax.set_xlabel(
        "Significant cluster-term pairs per replicate", fontsize=AXIS_LABEL_FONTSIZE, fontname=FONT
    )
    ax.set_ylabel("Null replicates (log scale)", fontsize=AXIS_LABEL_FONTSIZE, fontname=FONT)
    ax.set_xticks(x_transform(x_tick_values))
    ax.set_xticklabels([str(v) for v in x_tick_values], fontsize=TICK_LABEL_FONTSIZE, fontname=FONT)
    ax.set_xlim(XLIM_LEFT, obs_x + XLIM_RIGHT_PAD)
    ax.set_yscale("log")
    ax.set_ylim(YLIM_BOTTOM, YLIM_TOP)
    ax.grid(axis="y", color=GRID_COLOR, linewidth=GRID_LINEWIDTH, alpha=GRID_ALPHA, which="major")
    ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.spines["left"].set_color(SPINE_COLOR)
    ax.spines["bottom"].set_color(SPINE_COLOR)
    ax.spines["left"].set_linewidth(SPINE_LINEWIDTH)
    ax.spines["bottom"].set_linewidth(SPINE_LINEWIDTH)
    ax.tick_params(axis="both", labelsize=TICK_PARAMS_LABELSIZE)

In [ ]:
fig_a, ax_a = plt.subplots(figsize=PANEL_FIGSIZE)
fig_a.patch.set_facecolor("white")

plot_null_panel(ax_a, null_panels[0])

fig_a.tight_layout()
save_figure_png("panel_a_permutation_null", fig=fig_a)
plt.show()

In [ ]:
fig_b, ax_b = plt.subplots(figsize=PANEL_FIGSIZE)
fig_b.patch.set_facecolor("white")

plot_null_panel(ax_b, null_panels[1])

fig_b.tight_layout()
save_figure_png("panel_b_random_cluster_null", fig=fig_b)
plt.show()

In [ ]:
# Computed numeric summary for the caption below.
perm_value_counts = perm_null_dist["n_sig"].value_counts().sort_index()
rand_value_counts = rand_null_dist["n_sig"].value_counts().sort_index()

print(f"Observed significant cluster-term pairs (q<={QVAL_CUTOFF}): {n_observed_sig}")

print(f"\nAnnotation-label permutation null ({N_REPLICATES} replicates, seed={PERMUTATION_SEED}):")
for n_sig, count in perm_value_counts.items():
    print(f"  {count} replicates produced {n_sig} significant pair(s)")
print(f"  null max: {perm_row['null_max']}")
print(f"  empirical p-value: {perm_row['empirical_pvalue']:.6f}")

print(f"\nSame-size random-cluster null ({N_REPLICATES} replicates, seed={RANDOM_CLUSTER_SEED}):")
for n_sig, count in rand_value_counts.items():
    print(f"  {count} replicates produced {n_sig} significant pair(s)")
print(f"  null max: {rand_row['null_max']}")
print(f"  empirical p-value: {rand_row['empirical_pvalue']:.6f}")

## Caption / Claim Boundary

**Supplementary Figure S3. Null validation of yeast GI-PCC annotations.** The observed reference
clustering's significant cluster-term pair count (q≤0.05, printed above) lies far outside both
null distributions, each built from 1000 replicates and shown on a log-scaled count axis: an
annotation-label permutation null (clusters fixed, gene-to-term labeling shuffled) and a
same-size random-cluster null (annotations fixed, gene-to-cluster membership shuffled while
preserving cluster sizes). Both nulls rerun the same enrichment test and global Benjamini-Hochberg
FDR correction as the reference analysis, once per replicate, without rerunning hierarchical
clustering. Exact per-replicate value counts, null maxima, and empirical p-values (computed as
`((null_counts >= observed).sum() + 1) / (N_REPLICATES + 1)`) are printed immediately above. The
x-axis uses log10(count + 1) so the zero-heavy null distributions and the observed value can be
displayed together without an axis break.

These two controls address complementary arbitrariness concerns: whether enrichment could arise
from arbitrary annotation labels, and whether it could arise from arbitrary same-size cluster
partitions. Both nulls are scoped to the manual yeast GI-PCC reference configuration
(`linkage_threshold=16`, `min_cluster_size=30`) used throughout this figure package. This result
does not claim the reference clustering is optimal, does not claim biological completeness, and
does not claim superiority over any alternative clustering method; it shows only that the observed
enrichment signal is not reproduced by either targeted null control.